# 🗄️ SPARK TUTORIAL 03: SPARK SQL

## 🎯 **OBJETIVO**
Dominar Spark SQL para consultas complejas

## 📋 **CONTENIDO**
- Creación de vistas temporales
- Consultas SQL complejas
- Funciones SQL avanzadas
- Integración con Hive
- Optimización de consultas

---

## 🔧 **CONFIGURACIÓN INICIAL**


In [ ]:
# Importar librerías para Spark SQL
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

print("📚 Librerías de Spark SQL importadas correctamente")


In [ ]:
# Crear SparkSession con soporte SQL y Hive
import socket
import os

# Detectar si estamos dentro de un contenedor Docker
def get_spark_master():
    try:
        hostname = socket.gethostname()
        if 'jupyter' in hostname or 'master' in hostname:
            return "spark://master:7077"  # Desde dentro del contenedor
        else:
            return "spark://localhost:7077"  # Desde fuera del contenedor
    except:
        return "local[*]"  # Fallback a modo local

spark_master_url = get_spark_master()
print(f"🔧 Conectando a: {spark_master_url}")

spark = SparkSession.builder \
    .appName("EducacionIT-Spark-SQL") \
    .master(spark_master_url) \
    .config("spark.executor.memory", "800m") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.instances", "2") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("✅ SparkSession creada con soporte SQL y Hive")


## 📊 **PASO 1: CREAR DATOS DE EJEMPLO**

Vamos a crear tablas de ejemplo para consultas SQL.


In [ ]:
# Crear DataFrame de clientes
clientes_data = [
    (1, "Juan Pérez", "juan.perez@email.com", "Madrid", "2020-01-15", "Premium"),
    (2, "María García", "maria.garcia@email.com", "Barcelona", "2019-03-20", "Standard"),
    (3, "Carlos López", "carlos.lopez@email.com", "Madrid", "2021-06-10", "Premium"),
    (4, "Ana Martínez", "ana.martinez@email.com", "Valencia", "2018-11-05", "Basic"),
    (5, "Luis Rodríguez", "luis.rodriguez@email.com", "Sevilla", "2017-09-12", "Premium")
]

clientes_schema = StructType([
    StructField("cliente_id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("email", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("fecha_registro", StringType(), True),
    StructField("tipo_cliente", StringType(), True)
])

df_clientes = spark.createDataFrame(clientes_data, clientes_schema)
df_clientes = df_clientes.withColumn("fecha_registro", col("fecha_registro").cast(DateType()))

print("👥 Tabla clientes:")
df_clientes.show()


## 👁️ **PASO 2: CREAR VISTAS TEMPORALES**

Las vistas temporales permiten usar DataFrames como tablas SQL.


In [ ]:
# Crear vista temporal
df_clientes.createOrReplaceTempView("clientes")

print("✅ Vista temporal 'clientes' creada")
print("\n📋 Tablas disponibles:")
spark.sql("SHOW TABLES").show()


## 🔍 **PASO 3: CONSULTAS SQL BÁSICAS**

Vamos a ejecutar consultas SQL directamente.


In [ ]:
# Consulta simple
print("1️⃣ Todos los clientes premium:")
spark.sql("""
    SELECT cliente_id, nombre, ciudad, fecha_registro
    FROM clientes 
    WHERE tipo_cliente = 'Premium'
    ORDER BY fecha_registro DESC
""").show()


In [ ]:
# Análisis por ciudad
print("2️⃣ Clientes por ciudad:")
spark.sql("""
    SELECT ciudad, 
           COUNT(*) as total_clientes,
           COUNT(CASE WHEN tipo_cliente = 'Premium' THEN 1 END) as clientes_premium
    FROM clientes
    GROUP BY ciudad
    ORDER BY total_clientes DESC
""").show()


## 🎯 **RESUMEN DEL TUTORIAL**

¡Felicitaciones! Has completado el tutorial de Spark SQL.

### **📚 Conceptos SQL aprendidos:**
- ✅ **Vistas temporales**: Crear tablas SQL desde DataFrames
- ✅ **Consultas SQL básicas**: SELECT, WHERE, GROUP BY
- ✅ **Funciones SQL**: COUNT, CASE WHEN
- ✅ **Integración con Hive**: Soporte completo para Hive
- ✅ **Optimización**: Configuración para consultas eficientes

### **🚀 Próximos pasos:**
1. **Experimentar** con consultas más complejas
2. **Integrar** con datos reales del proyecto
3. **Optimizar** consultas de producción

---

**🎉 ¡Has dominado Spark SQL!**


In [ ]:
# Cerrar SparkSession
spark.stop()
print("🔒 SparkSession cerrada correctamente")
